In [1]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled
from langchain_classic.text_splitter import RecursiveCharacterTextSplitter
from langchain_google_genai import ChatGoogleGenerativeAI,GoogleGenerativeAIEmbeddings
from langchain_classic.vectorstores import FAISS
from langchain_core.prompts import PromptTemplate

In [2]:
llm=ChatGoogleGenerativeAI(model="gemini-2.5-flash",temperature=0.2)
embedding = GoogleGenerativeAIEmbeddings(model="models/gemini-embedding-001")

# INDEXING

In [3]:
video_id = "yHX_PV6LCOk" # only the ID, not full URL
try:
    ytApi=YouTubeTranscriptApi()
    transcript_list = ytApi.fetch(video_id, languages=["hi"])

    # Flatten it to plain text
    transcript = " ".join(chunk.text for chunk in transcript_list)
    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")
except Exception as e:
    print(f"An error occurred: {e}")

आखिर 15 दिन बाद संसद के सत्र में क्या होगा? 15 दिन बाद शुरू होने वाला संसद का सत्र क्यों विपक्ष और सरकार दोनों के लिए बहुत ही निर्णायक रहने वाला है? या फिर यह कहे कि इस संसद के सत्र के बाद से ही देश की राजनीति को एक बिल्कुल नई दिशा देने की कोशिश की जाएगी दोनों ही पक्षों द्वारा। दोस्तों जिस तारीख का इंतजार लंबे वक्त से था आखिरकार वो तारीख आ गई है। किरण रिजीजू ने अभी थोड़ी देर पहले ये जानकारी दी है कि 20 जुलाई से संसद का मानसून सत्र शुरू हो रहा है। दोस्तों ये जो मानसून सत्र है बहुत ही निर्णायक बहुत ही महत्वपूर्ण रहने वाला है क्योंकि ना केवल 13 अगस्त तक चलने वाले इस मानसून सत्र में काफी इंपॉर्टेंट और निर्णायक बिल सरकार लाने का प्लान बना रही है बल्कि ये वो सत्र होगा जिसमें सरकार को बहुत बुरी तरह से घेरने की जोर आजमाइश और कोशिश करेगा विपक्ष किस मुद्दे पर आप जानते हैं मुद्दों की भरमार है उनमें सबसे बड़ा मुद्दा सरकार के लिए परेशानी खड़े करने वाला है दोस्तों राम मंदिर में हुई चढ़ावे में चोरी का मुद्दा जिसको लेकर हर दिन नए खुलासे हो रहे हैं। खुद बीजेपी के लोग अबगाहे बगाहे या फिर खुलेआम वो बातें

# INDEXING
### Text Splitting

In [4]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])
print(len(chunks))

11


### Embedding Generation and Storing in Vector Store

In [5]:
vector_store = FAISS.from_documents(chunks, embedding)

# RETRIEVAL

In [6]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})
retriever.invoke("what is this about")

[Document(id='33c98523-64ad-41fb-951d-cc129e302795', metadata={}, page_content='सरकार ने कहा कि महिलाओं को आरक्षण देना है। इसीलिए संशोधन करना है और संशोधन करना पड़ेगा। इसीलिए परिसीमन का बिल लाना पड़ेगा। माना यह जा रहा है कि वो जो बिल गिर गया था एकद महीने पहले उसको दोबारा लाया जा सकता है। इस बार सरकार उसमें 50% वाली चीज लिख सकती है। क्योंकि पिछली बार मौखिक तौर पर कहा गया था अमित शाह द्वारा कि हर राज्य में 50% सीटें बढ़ेंगी। इस बार माना जा रहा है कि परिसीमन वाला बिल आ जाएगा जिसमें लोकसभा में सांसदों की संख्या 850 हो जाएगी। अगर वह बिल पास हो जाता है तो। 850 मतलब हर राज्य में 50% बढ़ेंगी तो 850 हो जाएगी। लोकसभा में सीटों की संख्या जो कि अभी 543 है। इसके अलावा महिला आरक्षण बिल तो खैर लाया ही जाएगा क्योंकि उसी की आड़ में तो यह परिसीमन बिल पास करवागी सरकार। इसके अलावा एक देश एक कानून जी हां दोस्तों ये जो एक देश एक कानून बिल है ना ये अ कमेटी के पास है और जिस तरह के बयान इन दिनों आ रहे हैं कमेटी से ऐसा माना जा रहा है कि इस बार के लोकसभा के सत्र में इसे पेश कर दिया जाएगा पास चाहे सरकार बाद में क

# AUGMENTATION

In [7]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
      Answer ONLY from the provided transcript context.
      If the context is insufficient, just say you don't know.

      {context}
      Question: {question}
    """,
    input_variables = ['context', 'question']
)

In [8]:
question  = "is the topic of nuclear fusion discussed in this video? If no they what is discussed in this"
retrieved_docs  = retriever.invoke(question)
print(retrieved_docs)

[Document(id='496362ce-223a-4839-85c4-05e5c52c3bcc', metadata={}, page_content='कोशिश करती थी क्योंकि उनके पास स्ट्रेंथ अब स्ट्रेंथ और कम हुई है। तो पहले भी सरकार जब नहीं सुनती थी और बहस नहीं कराती थी तो अब तो क्या ही किसी मुद्दे पर होगी बहस। हां लेकिन यह बात सही है कि इस बार का जो मानसून सत्र है वो रहने वाला बहुत हंगामेदार है। आपको क्या लगता है दोस्तों? कमेंट बॉक्स में जरूर लिखें और हमेशा की तरह संसद सत्र में जो भी अपडेट होगा दोस्तों तो आपको चैनल में जरूर देखने को मिलेगा।'), Document(id='6645edc0-0b4a-4dc7-82ae-62ede3708d95', metadata={}, page_content='आती है तो मैं आपको जानकारी दूंगी। मैं नहीं चाहती कि कोई मैं ऐसी जानकारी आपको दूं जिसको लेकर मैं खुद कंफर्म नहीं हूं। एक और बहुत निर्णायक बिल है जिसका पिछली बार सरकार जिसको पास नहीं करवा पाई थी। सरकार को मुगी खानी पड़ी थी उस बिल में। तो हो सकता है कि उस बिल को लेकर भी सरकार इस बार हिम्मत दिखाए। तो ये बहुत ही आप समझ लीजिए बहुत ही कंट्रोवर्शियल बिल है। परिसीमन बिल अभी गिरा है सरकार का 2 महीने पहले। ठीक है? ये बंगाल के चुनाव से पहले और अब त

In [9]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
final_prompt = prompt.invoke({"context": context_text, "question": question})

# GENERATION

In [10]:
answer = llm.invoke(final_prompt)
print(answer.content)

No, the topic of nuclear fusion is not discussed in this video.

Instead, the transcript discusses several other topics, including:
*   The upcoming monsoon session of Parliament and its expected tumultuous nature.
*   The delimitation bill and its potential impact on Lok Sabha seats and national politics.
*   The 'One Nation One Election' bill and the need for constitutional amendments.
*   The Judicial Constitutional Reform Bill, which includes UCC and other reforms.
*   The issue of theft in Ram Mandir donations.
*   The ongoing CJP protest and Sonam Wangchuk's health.


# BUILDING CHAIN

In [11]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [12]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [13]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [14]:
parallel_chain.invoke('who is Demis')

{'context': 'का और अगर वो समर्थन में आ जाती है तो फिर बात अलग ही है। डीएमके को लेकर चर्चाएं हैं कि डीएमके कुछ इशू बेस्ड मुद्दों पर सरकार को सपोर्ट कर सकती है और ये जो परिसीमन वाला बिल है उनमें से एक हो सकता है। अब कुल जमा बात यह है कि ऑलमोस्ट 15 दिन बाद पता चलेगा कि देश की राजनीति की दिशा और दशा किस ओर बढ़ती है। क्या सरकार को चंदा चोरी मामले में लेने के देने पड़ सकते हैं? विपक्ष अपनी मजबूती से अपनी आवाज उठा पाता है क्योंकि विपक्ष की आंकड़ों के तहत मजबूती काफी कम हुई है। टीएमसी टूट चुकी है। उद्धव ठाकरे की पार्टी कमजोर हो चुकी है। अब सरकार के सामने खड़े होने की स्ट्रेंथ भी कमजोर हुई है विपक्ष की। तो जब आपकी स्ट्रेंथ कमजोर होती है तो फिर जाहिर तौर पर आपकी बारगेनिंग पावर भी कमजोर होती है। तो पहले अगर सरकार के आगे विपक्ष हंगामा करता था और सरकार से बहस की मांग करता था किसी मुद्दे पर तो सरकार उनकी सुनने की कोशिश करती थी क्योंकि उनके पास स्ट्रेंथ अब स्ट्रेंथ और कम हुई है। तो पहले भी सरकार जब नहीं सुनती थी और बहस नहीं कराती थी तो अब तो क्या ही किसी मुद्दे पर होगी बहस। हां लेकिन यह बात सही है कि

In [15]:
parser = StrOutputParser()
main_chain = parallel_chain | prompt | llm | parser

In [17]:
main_chain.invoke('Can you summarize the video in english')

'The provided transcript discusses several key issues and predictions for the upcoming parliamentary session:\n\n1.  **Ram Mandir Donation Theft:** This is a major problem for the government, with daily revelations and even BJP members speaking out. The opposition plans to use this issue to derail the government\'s agenda.\n2.  **CJP Protest:** The Cockro Janata Party\'s protest over paper leaks is ongoing, with support from Congress (Rahul Gandhi\'s party) and TMC (Mahua Moitra). Concerns are raised about Sonam Wangchuk\'s health.\n3.  **Monsoon Session:** The upcoming monsoon session is expected to be very tumultuous, with the opposition focusing on the donation theft to divert attention from government bills.\n4.  **Delimitation Bill:** A highly controversial and decisive Delimitation Bill, which the government previously failed to pass, might be reintroduced. If passed, it could significantly alter Indian politics by increasing Lok Sabha seats (potentially by 50%) and disproportion